# ASSIGNMENT: Cartoon Face Mask
## Task 6 — Ankit Jhurani

This assignment demonstrates:
- **Face detection** using OpenCV Haar Cascade classifiers
- **Image masking** using alpha-channel blending
- **Image thresholding** concepts for mask transparency
- **Video/image processing** pipeline
- Cartoon mask overlay on both **static images** and **live webcam feed**

## Step 0: Import Libraries

In [ ]:
import cv2
import numpy as np
import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

print(f"OpenCV version: {cv2.__version__}")
print("All libraries loaded successfully!")

## Step 1: Create the Cartoon Mask

We create a **SpongeBob-inspired** cartoon character face using PIL's drawing tools.
The mask uses an **RGBA** format where the alpha channel controls transparency — areas
outside the face shape are fully transparent (alpha=0), and the cartoon features are opaque (alpha=255).

In [ ]:
def create_cartoon_mask(size=400):
    """
    Creates a cartoon character face mask with RGBA transparency.
    
    The alpha channel acts as a binary/soft threshold mask:
      - alpha = 0   → fully transparent (background shows through)
      - alpha = 255 → fully opaque (cartoon overlays the face)
    
    Returns: PIL Image in RGBA mode
    """
    img = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    
    cx, cy = size // 2, size // 2

    # --- Head: yellow circular face ---
    head_color = (255, 220, 50, 255)
    draw.ellipse([4, 4, size - 4, size - 4],
                 fill=head_color, outline=(180, 140, 0, 255), width=4)

    # --- Eyes: white circles with blue pupils ---
    eye_r = size // 8
    eye_y = cy - size // 10
    for ex in [cx - size // 3, cx + size // 3]:
        # White of eye
        draw.ellipse([ex - eye_r, eye_y - eye_r, ex + eye_r, eye_y + eye_r],
                     fill=(255, 255, 255, 255), outline=(0, 0, 0, 255), width=2)
        # Pupil
        pr = eye_r // 3
        draw.ellipse([ex - pr, eye_y - pr, ex + pr, eye_y + pr],
                     fill=(30, 100, 200, 255))
        # Shine dot
        sr = pr // 2
        draw.ellipse([ex - sr + pr // 2, eye_y - sr - pr // 2,
                      ex + sr // 2 + pr // 2, eye_y + sr // 2 - pr // 2],
                     fill=(255, 255, 255, 200))

    # --- Eyebrows ---
    brow_y = eye_y - eye_r - 10
    draw.line([cx - size // 3 - eye_r, brow_y,
               cx - size // 3 + eye_r, brow_y - 14],
              fill=(80, 50, 10, 255), width=7)
    draw.line([cx + size // 3 - eye_r, brow_y - 14,
               cx + size // 3 + eye_r, brow_y],
              fill=(80, 50, 10, 255), width=7)

    # --- Nose ---
    nw, nh = size // 12, size // 10
    draw.ellipse([cx - nw, cy - nh // 2, cx + nw, cy + nh // 2],
                 fill=(220, 160, 50, 255), outline=(160, 100, 20, 255), width=2)

    # --- Smile arc ---
    smile_box = [cx - size // 3, cy + size // 12,
                 cx + size // 3, cy + size // 3]
    draw.arc(smile_box, start=0, end=180, fill=(0, 0, 0, 255), width=4)

    # --- Teeth (2 rectangles inside smile) ---
    ty1 = cy + size // 12 + 6
    ty2 = ty1 + size // 8
    for i in range(2):
        tx = cx - size // 6 + i * (size // 6)
        draw.rectangle([tx + 2, ty1, tx + size // 6 - 2, ty2],
                       fill=(255, 255, 255, 255), outline=(200, 200, 200, 255), width=1)

    # --- Freckles ---
    fr = size // 20
    for fx, fy in [(cx - size // 3, cy + size // 16),
                   (cx + size // 3, cy + size // 16)]:
        draw.ellipse([fx - fr, fy - fr // 2, fx + fr, fy + fr // 2],
                     fill=(200, 150, 50, 180))

    return img


# Generate and save the mask
cartoon_mask = create_cartoon_mask(400)
cartoon_mask.save("cartoon_mask.png")

# Display
plt.figure(figsize=(4, 4))
plt.imshow(cartoon_mask)
plt.axis('off')
plt.title('Cartoon Mask (SpongeBob-style)', fontsize=14)
plt.tight_layout()
plt.show()
print("Cartoon mask saved as 'cartoon_mask.png'")

## Step 2: Face Detection Setup

We use OpenCV's **Haar Cascade classifier** (`haarcascade_frontalface_alt2`) for face detection.
This is a pre-trained model that uses a sliding window of feature detectors across the image.

In [ ]:
# Load Haar Cascade classifiers (bundled with OpenCV)
cascPathface = os.path.dirname(cv2.__file__) + "/data/haarcascade_frontalface_alt2.xml"
cascPatheyes = os.path.dirname(cv2.__file__) + "/data/haarcascade_eye_tree_eyeglasses.xml"

faceCascade = cv2.CascadeClassifier(cascPathface)
eyeCascade  = cv2.CascadeClassifier(cascPatheyes)

print("Face cascade loaded:", not faceCascade.empty())
print("Eye cascade loaded: ", not eyeCascade.empty())

## Step 3: Alpha-Blending Overlay Function

The cartoon mask is composited onto the frame using **alpha blending**:

$$\text{output} = \alpha \cdot \text{mask} + (1 - \alpha) \cdot \text{frame}$$

where $\alpha \in [0, 1]$ comes from the mask's alpha channel.

In [ ]:
def overlay_cartoon_mask(frame, faces, mask_pil):
    """
    Overlays a cartoon RGBA mask over each detected face region.

    Parameters
    ----------
    frame    : np.ndarray  — BGR image (OpenCV format)
    faces    : np.ndarray  — array of (x, y, w, h) face bounding boxes
    mask_pil : PIL.Image   — RGBA cartoon mask

    Returns
    -------
    np.ndarray  — frame with cartoon masks composited over faces
    """
    result   = frame.copy()
    mask_arr = np.array(mask_pil.convert('RGBA'), dtype=np.float32)

    for (x, y, w, h) in faces:
        # Resize mask to exactly fit the bounding box
        mask_resized = cv2.resize(mask_arr, (w, h))

        # Normalise alpha to [0, 1]
        alpha = mask_resized[:, :, 3:4] / 255.0          # shape (h, w, 1)

        # Cartoon RGB (note: PIL is RGB, OpenCV is BGR — swap channels)
        cartoon_bgr = mask_resized[:, :, [2, 1, 0]]      # RGB → BGR

        roi = result[y:y + h, x:x + w].astype(np.float32)

        # Alpha composite
        blended = alpha * cartoon_bgr + (1.0 - alpha) * roi
        result[y:y + h, x:x + w] = blended.astype(np.uint8)

    return result


def detect_faces(frame, scale_factor=1.1, min_neighbors=5, min_size=60):
    """
    Detects faces in a BGR frame and returns bounding boxes.
    Picks only the largest detected face to avoid false positives.
    """
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = faceCascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=(min_size, min_size),
        flags=cv2.CASCADE_SCALE_IMAGE
    )
    if len(faces) == 0:
        return np.array([])
    # Return only the largest face box
    largest = max(faces, key=lambda f: f[2] * f[3])
    return np.array([largest])


print("Helper functions defined.")

## Step 4: Apply Cartoon Mask to Static Images

Since live webcam capture isn't available in this notebook run, we demonstrate the full pipeline
on four reference photos.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
# (scale_factor, min_neighbors, min_size_px) tuned per image set
image_configs = [
    ("IMG_0218.jpeg", 1.10, 5, 60),   # indoor close-up
    ("IMG_1430.jpeg", 1.10, 5, 60),   # indoor close-up
    ("IMG_5972.jpeg", 1.05, 3, 200),  # outdoor balcony
    ("IMG_7110.jpeg", 1.05, 3, 200),  # outdoor balcony
]

results_display = []

for fname, sf, mn, ms in image_configs:
    img_path = fname  # adjust path if needed
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"⚠  Could not read {fname}")
        continue

    faces = detect_faces(frame, scale_factor=sf, min_neighbors=mn, min_size=ms)
    out   = overlay_cartoon_mask(frame, faces, cartoon_mask)

    # Save output
    out_name = f"cartoon_{fname.replace('.jpeg', '.jpg')}"
    cv2.imwrite(out_name, out)

    results_display.append((fname, frame, faces, out))
    print(f"{fname}: {len(faces)} face(s) detected → saved {out_name}")

print("\nAll images processed!")

## Step 5: Visualise Results (Before vs After)

In [ ]:
fig, axes = plt.subplots(len(results_display), 2, figsize=(12, 6 * len(results_display)))

for i, (fname, original, faces, result) in enumerate(results_display):
    # Draw green bounding box on original
    orig_vis = original.copy()
    for (x, y, w, h) in faces:
        cv2.rectangle(orig_vis, (x, y), (x + w, y + h), (0, 255, 0), 4)

    axes[i][0].imshow(cv2.cvtColor(orig_vis, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f"Original + Face Detection\n{fname}", fontsize=11)
    axes[i][0].axis('off')

    axes[i][1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    axes[i][1].set_title(f"Cartoon Mask Applied\n{fname}", fontsize=11)
    axes[i][1].axis('off')

plt.suptitle("Task 6 — Cartoon Face Mask Results", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("cartoon_results_grid.png", dpi=100, bbox_inches='tight')
plt.show()
print("Grid saved as 'cartoon_results_grid.png'")

## Step 6: Live Webcam Feed with Cartoon Mask + Video Save

The code below runs the real-time webcam pipeline.

> **To run**: Execute this cell in a local Jupyter environment with a webcam connected.  
> Press **`q`** to stop recording and save the video.

The video is written to `cartoon_webcam_output.avi` using the XVID codec.

In [ ]:
# ─── LIVE WEBCAM CARTOON MASK + VIDEO RECORDING ────────────────────────────
# Requires: physical webcam, local Jupyter environment
# Press 'q' to quit and save the video.

OUTPUT_VIDEO = "cartoon_webcam_output.avi"
FOURCC       = cv2.VideoWriter_fourcc(*'XVID')
FPS          = 20
FRAME_SIZE   = (640, 480)

video_capture = cv2.VideoCapture(0)

if not video_capture.isOpened():
    print("⚠  Could not open webcam. Run on a machine with a connected camera.")
else:
    out_writer = cv2.VideoWriter(OUTPUT_VIDEO, FOURCC, FPS, FRAME_SIZE)
    print("Recording started. Press 'q' to stop...")

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        # Resize to target resolution
        frame = cv2.resize(frame, FRAME_SIZE)

        # Detect face
        faces = detect_faces(frame, scale_factor=1.1, min_neighbors=5, min_size=60)

        # Apply cartoon mask
        if len(faces) > 0:
            frame = overlay_cartoon_mask(frame, faces, cartoon_mask)

        # Write frame to video file
        out_writer.write(frame)

        # Display live feed
        cv2.imshow('Cartoon Face Mask — press q to stop', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    video_capture.release()
    out_writer.release()
    cv2.destroyAllWindows()
    print(f"Video saved to '{OUTPUT_VIDEO}'")

## Step 7: Summary

| Concept | How it was applied |
|---|---|
| **Face detection** | `haarcascade_frontalface_alt2` Haar cascade classifier |
| **Image masking** | RGBA alpha channel used to define transparent cartoon regions |
| **Thresholding** | Alpha channel acts as a soft threshold: 0 = background, 255 = cartoon |
| **Alpha blending** | `output = α·mask + (1−α)·frame` for seamless compositing |
| **Video capturing** | `cv2.VideoCapture(0)` reads from webcam |
| **Video saving** | `cv2.VideoWriter` with XVID codec saves to `.avi` |

### Files submitted
1. `Ankit_Task6_Cartoon_Filter.ipynb` — this notebook  
2. `cartoon_mask.png` — the cartoon character mask  
3. `cartoon_results_grid.png` — screenshot showing mask applied to 4 photos  
4. `cartoon_webcam_output.avi` — recorded video (run Step 6 locally to generate)